# Rollout比較：MSEと相対L2
各stepでReケースを等しい重みで平均します。内挿と外挿は別々に表示します。
既存CSVのみを読み、学習・推論は実行しません。欠損CSVは一覧に表示します。

誤差は物理スケール・流体領域の値です。相対L2はケースごとの比の算術平均です。
overall MSEは異なる単位のu/v/pを含む既存指標なので、成分別も確認してください。
曲線は時刻ごとの誤差であり、時間方向の累積和ではありません。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "performance/results").is_dir())
EXPERIMENTS = ["ex005", "ex006", "ex007", "ex008"]
SPLITS = ["interpolation", "extrapolation"]
LOG_SCALE = False  # 小さい値を比較するときはTrue
SAVE_RESULTS = False  # Trueで比較図と平均CSVを保存
OUTPUT_DIR = ROOT / "performance/results/rollout_comparison"
FIELDS = ["overall", "u", "v", "p"]
METRICS = [f"{m}_{f}" for m in ["mse", "relative_l2"] for f in FIELDS]


In [ ]:
data, availability = {}, []
for split in SPLITS:
    for experiment in EXPERIMENTS:
        path = ROOT / "performance/results" / f"{experiment}_baseline" / split / "rollout/rollout_metrics.csv"
        if not path.is_file():
            availability.append(dict(split=split, experiment=experiment, status="CSV missing"))
            continue
        df = pd.read_csv(path)
        required = ["Re", "step", "time", "elapsed_time", *METRICS]
        if not set(required).issubset(df.columns):
            raise ValueError(f"{path}: required columns missing")
        if df.empty or not np.isfinite(df[required].to_numpy(dtype=float)).all():
            raise ValueError(f"{path}: empty data or NaN/Inf")
        if df.duplicated(["Re", "step"]).any() or (df[METRICS] < 0).any().any():
            raise ValueError(f"{path}: duplicate Re/step or negative error")
        df = df.sort_values(["Re", "step"]).reset_index(drop=True)
        expected = pd.MultiIndex.from_product([sorted(df.Re.unique()), sorted(df.step.unique())])
        if not pd.MultiIndex.from_frame(df[["Re", "step"]]).equals(expected):
            raise ValueError(f"{path}: missing Re/step combinations")
        for _, group in df.groupby("step"):
            times = group[["time", "elapsed_time"]].to_numpy()
            if not np.allclose(times, times[0], rtol=0, atol=1e-6):
                raise ValueError(f"{path}: time differs across Re")
        data[(split, experiment)] = df
        availability.append(dict(split=split, experiment=experiment, status="available",
                                 Re=sorted(df.Re.unique()), steps=df.step.nunique()))
display(pd.DataFrame(availability))

for split in SPLITS:
    reference = None
    for experiment in EXPERIMENTS:
        if (split, experiment) not in data:
            continue
        keys = data[(split, experiment)][["Re", "step", "time", "elapsed_time"]].to_numpy()
        if reference is not None and (keys.shape != reference.shape or
                not np.allclose(keys, reference, rtol=0, atol=1e-6)):
            raise ValueError(f"{split}: Re/step/time differs for {experiment}")
        reference = keys
means = {key: df.groupby("step", as_index=False)[["time", "elapsed_time", *METRICS]].mean()
         for key, df in data.items()}
if not means:
    raise ValueError("No rollout CSV found")


## 平均誤差の時間変化
上段：MSE、下段：相対L2。列：overall・u・v・p。
CSVを追加したら上から再実行すると比較対象に入ります。


In [ ]:
colors = dict(zip(EXPERIMENTS, plt.get_cmap("tab10").colors))
for split in SPLITS:
    available = [e for e in EXPERIMENTS if (split, e) in means]
    if not available:
        continue
    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True, squeeze=False)
    for row, metric in enumerate(["mse", "relative_l2"]):
        for col, field in enumerate(FIELDS):
            ax = axes[row, col]
            for experiment in available:
                mean = means[(split, experiment)]
                ax.plot(mean.elapsed_time, mean[f"{metric}_{field}"],
                        label=experiment.upper(), color=colors[experiment])
            ax.set_title(field)
            ax.set_ylabel("Mean MSE" if metric == "mse" else "Mean relative L2")
            ax.set_xlabel("Elapsed time [s]")
            ax.grid(alpha=0.3)
            if LOG_SCALE:
                ax.set_yscale("log")
    axes[0, 0].legend()
    fig.suptitle(f"Rollout comparison: {split} (equal weight per Re)")
    fig.tight_layout()
    if SAVE_RESULTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / f"{split}_mse_relative_l2.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 要約表
final：最終step、time_mean：各stepの単純平均、peak：Re平均曲線の最大値。
個別Reの最大誤差ではありません。


In [ ]:
rows = []
for (split, experiment), mean in means.items():
    for metric in METRICS:
        rows.append(dict(split=split, experiment=experiment, metric=metric,
                         Re_count=data[(split, experiment)].Re.nunique(), steps=len(mean),
                         final=mean[metric].iloc[-1], time_mean=mean[metric].mean(),
                         peak=mean[metric].max()))
    if SAVE_RESULTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        mean.to_csv(OUTPUT_DIR / f"{experiment}_{split}_mean.csv", index=False)
summary = pd.DataFrame(rows)
for split in SPLITS:
    for metric in ["mse", "relative_l2"]:
        subset = summary[(summary.split == split) & summary.metric.str.startswith(metric + "_")]
        if not subset.empty:
            print(f"{split} / {metric}")
            display(subset.pivot(index="experiment", columns="metric", values=["final", "time_mean", "peak"]))
if SAVE_RESULTS:
    summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
    print(f"Saved: {OUTPUT_DIR}")
